# Import and Read data


In [23]:
import sys
sys.path.append('.')  # Add current directory to path
from haversine_build_graph_and_train import *
import torch

In [24]:
partition = 200

In [25]:
df = pd.read_csv(f"../../../data/top30groups/LongLatCombined/combined/combined{partition}.csv")

In [26]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [27]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [28]:
geodata = ['longitude', 'latitude']
combined_geo = df.copy()
combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
combined_geo = combined_geo.drop(columns=geodata)

In [29]:
import ast

def to_tuple_if_needed(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val  # already a tuple

combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [30]:
torch.cuda.empty_cache()


In [31]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
# Hyperparameter grid
param_grid = {
    'lr': [0.001],
    'n_tree': [100],
    'tree_depth': [10],
    'tree_feature_rate': [0.5],
    'feat_dropout': [0.1],
    'embed_dim': [32]
}

partitions =  [200, 300, 478]

# Convert to list of dicts (cartesian product)
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
for part in partitions:
    for col in continuous_cols:
        print(f"\nTraining model for {col} prediction...")

        y_nrf, nrf_input, train_df, val_df, test_df, row_to_node_index, index_to_label = build_graph_data(
            combined_geo, label_index, continuous_col=col)

        best_run = None
        best_score = -1

        for combo in grid_combos:
            args = {
                **dict(zip(param_names, combo)),
                'partition': f"gtd{part}",
                'n_class': len(label_index),
                'epochs': 3000,
                'final_evaluation': True
            }

            print(f"Running config: {args}")
            acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
                y_nrf, nrf_input, args, row_to_node_index, index_to_label, True, train_df, val_df, test_df)

            if acc > best_score:
                best_score = acc
                best_run = {
                    "args": args,
                    "acc": acc,
                    "epoch": epoch,
                    "y_pred": y_pred_decoded,
                    "y_true": y_true_decoded,
                    "precision": p,
                    "recall": r,
                    "f1": f1,
                    "micro": (p_micro, r_micro, f1_micro),
                    "macro": (p_macro, r_macro, f1_macro),
                    "auroc": (auc_w, auc_mi, auc_ma),
                    "epoch_logs": epoch_logs
                }


        # Save best results
        if best_run:
            args = best_run["args"]
            os.makedirs(f"Results{part}", exist_ok=True)

            results_path = f"Results{part}/Results_{col}_prediction"
            with open(results_path, "w") as f:
                f.write(f"Best acc: {best_run['acc']:.4f} at epoch {best_run['epoch']} for {col} prediction\n")
                f.write(f"Config: {args}\n")
                f.write(f"Weighted Precision: {best_run['precision']:.4f}, Recall: {best_run['recall']:.4f}, F1: {best_run['f1']:.4f}\n")
                f.write(f"Macro Precision: {best_run['macro'][0]:.4f}, Recall: {best_run['macro'][1]:.4f}, F1: {best_run['macro'][2]:.4f}\n")
                f.write(f"Micro Precision: {best_run['micro'][0]:.4f}, Recall: {best_run['micro'][1]:.4f}, F1: {best_run['micro'][2]:.4f}\n")
                f.write(f"AUROC Weighted: {best_run['auroc'][0]:.4f}, Micro: {best_run['auroc'][1]:.4f}, Macro: {best_run['auroc'][2]:.4f}\n")

            log_path = f"Results{part}/epoch_logs_{col}_prediction"
            with open(log_path, "w") as f:
                f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

            y_preds.append(best_run['y_pred'])
            y_trues.append(best_run['y_true'])

    print(best_score)


Training model for weaptype1 prediction...
Running config: {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd200', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
Epoch 000 | NRF Loss: 3.4012 | Joint: 3.4012 | Val Acc: 0.3375
Epoch 050 | NRF Loss: 3.0546 | Joint: 3.0546 | Val Acc: 0.5700
Epoch 100 | NRF Loss: 2.8697 | Joint: 2.8697 | Val Acc: 0.7267
Epoch 150 | NRF Loss: 2.7225 | Joint: 2.7225 | Val Acc: 0.7667
Epoch 200 | NRF Loss: 2.5894 | Joint: 2.5894 | Val Acc: 0.8025
Epoch 250 | NRF Loss: 2.4666 | Joint: 2.4666 | Val Acc: 0.8058
Epoch 300 | NRF Loss: 2.3526 | Joint: 2.3526 | Val Acc: 0.8233
Epoch 350 | NRF Loss: 2.2465 | Joint: 2.2465 | Val Acc: 0.8342
Epoch 400 | NRF Loss: 2.1461 | Joint: 2.1461 | Val Acc: 0.8425
Epoch 450 | NRF Loss: 2.0506 | Joint: 2.0506 | Val Acc: 0.8533
Epoch 500 | NRF Loss: 1.9611 | Joint: 1.9611 | Val Acc: 0.8567
Epoch 550 | NRF Loss: 1.8767 | Joint: 1.8767 | Val Acc: 

In [32]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [33]:
print(best_run)

{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd478', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9016667008399963, 'epoch': 2888, 'y_pred': ["Kurdistan Workers' Party (PKK)", 'Revolutionary Armed Forces of Colombia (FARC)', 'Basque Fatherland and Freedom (ETA)', 'Palestinians', 'Shining Path (SL)', 'Sikh Extremists', 'Irish Republican Army (IRA)', 'Shining Path (SL)', 'Fulani extremists', "Donetsk People's Republic", 'Maoists', 'Islamic State of Iraq and the Levant (ISIL)', 'Basque Fatherland and Freedom (ETA)', 'Al-Shabaab', 'Manuel Rodriguez Patriotic Front (FPMR)', 'Nicaraguan Democratic Force (FDN)', 'Basque Fatherland and Freedom (ETA)', 'Communist Party of India - Maoist (CPI-Maoist)', 'Maoists', 'Taliban', 'Tupac Amaru Revolutionary Movement (MRTA)', 'Al-Qaida in Iraq', 'Sikh Extremists', 'Muslim extremists', 'Taliban', 'Liberation Tigers of Tamil Eelam (LTTE)', 'Comm

In [34]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [35]:
best_acc

NameError: name 'best_acc' is not defined

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

'\nBest acc: 0.9205 at epoch 750 for weaptype1 prediction\nWeighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191\nMacro Precision: 0.9176, Recall: 0.9107, F1: 0.9105\nMicro Precision: 0.9205, Recall: 0.9205, F1: 0.9205\nAUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963\n\n'

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.98      0.93      0.95        43
        African National Congress (South Africa)       0.98      1.00      0.99        58
                                Al-Qaida in Iraq       1.00      0.79      0.88        87
        Al-Qaida in the Arabian Peninsula (AQAP)       0.98      0.92      0.95        50
                                      Al-Shabaab       1.00      1.00      1.00        58
             Basque Fatherland and Freedom (ETA)       0.98      0.90      0.94        60
                                      Boko Haram       0.93      0.95      0.94        41
  Communist Party of India - Maoist (CPI-Maoist)       1.00      0.92      0.96        24
       Corsican National Liberation Front (FLNC)       0.97      0.96      0.97        78
                       Donetsk People's Republic       1.00      1.00      1.00        59
Farabundo

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])

Saved confusion matrix for partition 100 to Results100/cm_100_weaptype1.png
